# Ames Housing - Exploratory Data Analysis (EDA)

در این نوت‌بوک، تحلیل اکتشافی داده‌های پروژه پیش‌بینی قیمت مسکن انجام می‌شود.
هدف این مرحله، شناخت ساختار داده، بررسی مقادیر گمشده، تحلیل متغیر هدف و پیدا کردن الگوهای مهم برای مرحله preprocessing و modeling است.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# تنظیمات نمایش برای خوانایی بهتر
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# تنظیم استایل نمودارها
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 1. Load Dataset


In [ ]:
# خواندن دیتاست
df = pd.read_csv("../data/raw/ames_housing.csv")

# نمایش 5 سطر اول
df.head()


## 2. Initial Inspection


In [ ]:
# تعداد سطر و ستون
df.shape


In [ ]:
# لیست ستون‌ها
df.columns.tolist()


In [ ]:
# نوع داده هر ستون
df.dtypes


In [ ]:
# اطلاعات کلی دیتاست
df.info()


In [ ]:
# آمار توصیفی ویژگی‌های عددی
df.describe()


In [ ]:
# آمار توصیفی ویژگی‌های دسته‌ای
df.describe(include="object")


## 3. Missing Values Analysis


In [ ]:
# تعداد مقادیر گمشده در هر ستون
missing_count = df.isnull().sum()

# فقط ستون‌هایی که missing دارند
missing_count = missing_count[missing_count > 0].sort_values(ascending=False)

missing_count


In [ ]:
# درصد مقادیر گمشده در هر ستون
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)

missing_percent


In [ ]:
# جدول کامل missing values
missing_data = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})

missing_data = missing_data[missing_data["missing_count"] > 0]
missing_data = missing_data.sort_values(by="missing_percent", ascending=False)

missing_data


In [ ]:
top_missing = missing_data.head(20)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_missing.index, y=top_missing["missing_percent"], palette="viridis")
plt.xticks(rotation=90)
plt.ylabel("Missing Percentage")
plt.xlabel("Features")
plt.title("Top 20 Features with Missing Values")
plt.show()


### جمع‌بندی اولیه Missing Values
- چندین ستون دارای missing values هستند.
- برخی از این ستون‌ها احتمالاً missing معنادار دارند و نشان‌دهنده نبود یک ویژگی در ملک هستند، نه نقص در ثبت داده.
- ستون‌هایی مانند ویژگی‌های basement، garage، alley و fireplace باید در preprocessing با دقت بیشتری بررسی شوند.
- برای تصمیم نهایی، در مرحله preprocessing ستون‌ها را به سه دسته تقسیم می‌کنیم:
  1. missing به معنی عدم وجود ویژگی
  2. missing قابل جایگزینی با median/mode
  3. ستون‌هایی که شاید حذف شوند


## 4. Target Variable Analysis


In [ ]:
df["SalePrice"].describe()


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["SalePrice"], kde=True)
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df["SalePrice"])
plt.title("Boxplot of SalePrice")
plt.xlabel("SalePrice")
plt.show()


In [ ]:
df["SalePrice"].skew()


### برداشت اولیه از SalePrice
- توزیع قیمت فروش را بررسی کردیم.
- اگر توزیع راست‌چوله (right-skewed) باشد، ممکن است در مرحله preprocessing از تبدیل لگاریتمی برای متغیر هدف استفاده کنیم.
- Boxplot به شناسایی مقادیر پرت در قیمت فروش کمک می‌کند.


## 5. Correlation Analysis of Numerical Features


In [ ]:
numeric_df = df.select_dtypes(include=["int64", "float64"])
corr_matrix = numeric_df.corr()

corr_matrix["SalePrice"].sort_values(ascending=False)


In [ ]:
top_corr = corr_matrix["SalePrice"].sort_values(ascending=False)
top_corr.head(15)


In [ ]:
top_features = top_corr.head(15).index
plt.figure(figsize=(12, 8))
sns.heatmap(df[top_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Top Correlated Numerical Features with SalePrice")
plt.show()


### جمع‌بندی اولیه همبستگی
- برخی ویژگی‌های عددی همبستگی بیشتری با `SalePrice` دارند.
- این ویژگی‌ها احتمالاً در مدل‌سازی نقش مهم‌تری خواهند داشت.
- همچنین ممکن است بین بعضی ویژگی‌ها همبستگی بالا وجود داشته باشد که در مدل‌های خطی باید به آن توجه کنیم.


## 6. Outlier Analysis


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df["Gr Liv Area"], y=df["SalePrice"])
plt.title("Gr Liv Area vs SalePrice")
plt.xlabel("Gr Liv Area")
plt.ylabel("SalePrice")
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df["Lot Area"], y=df["SalePrice"])
plt.title("Lot Area vs SalePrice")
plt.xlabel("Lot Area")
plt.ylabel("SalePrice")
plt.show()


### جمع‌بندی اولیه نقاط پرت
- نمودارهای پراکندگی برای شناسایی نقاط پرت در ویژگی‌های مهم عددی رسم شدند.
- اگر نمونه‌هایی با مساحت بسیار بالا اما قیمت غیرعادی دیده شوند، ممکن است در مرحله preprocessing نیاز به بررسی یا حذف آن‌ها باشد.
- تصمیم نهایی برای حذف outlierها باید با دقت و فقط بر اساس شواهد کافی انجام شود.


## 7. Final EDA Summary


### نتیجه‌گیری از EDA
- دیتاست شامل ویژگی‌های عددی و دسته‌ای متنوع است.
- تعداد قابل توجهی missing value در برخی ستون‌ها وجود دارد.
- متغیر هدف `SalePrice` باید از نظر چولگی و نرمال‌سازی بررسی شود.
- برخی ویژگی‌های عددی رابطه قوی‌تری با قیمت فروش دارند.
- وجود چند نقطه پرت محتمل است و در preprocessing دوباره بررسی خواهد شد.

### تصمیم برای مرحله بعد
در مرحله preprocessing باید:
1. ستون‌های دارای missing values را دسته‌بندی کنیم.
2. ستون‌های categorical را encode کنیم.
3. برخی ویژگی‌های عددی را scale یا transform کنیم.
4. درباره حذف یا نگه‌داشتن outlierها تصمیم بگیریم.
